# Pandas and the District Voter File


`pandas` is a library (i.e., a code collection) of useful functions and objects to work with data. It can read from a variety of formats (CSV, Excel, ...) and has a load of nifty tools to analyze and visualize data.

In this notebook we'll practice the basics of `pandas` on a **district voter file** — the kind of dataset a campaign or a turnout researcher works with every day: one row per voter, with demographic and vote-history information.


## Loading data

`pandas` can read *X*-separated files (using `read_csv(<NAME>, sep='X')`), such as CSV and TSV. The result is a new object, called a **`DataFrame`**. You can think of it as a table on steroids.


In [ ]:
# this just tells the notebook to display images in the cells, rather than send it to a new window
import numpy as np
import pandas as pd
import requests
from io import StringIO
import seaborn as sb
import matplotlib.pyplot as plt
sb.set_context('paper')
sb.set_style('dark')
%matplotlib inline

# URL of the CSV file
url = ('https://raw.githubusercontent.com/joshuakalla/data_science_campaigns_26/'
       'main/weeks/wk10_turnout_models/data/district_voter_file.csv')

# Request the content of the URL
response = requests.get(url)

# Check if the request was successful
if response.status_code == 200:
    # Use StringIO to read the CSV text into a pandas DataFrame
    csv_data = StringIO(response.text)
    voters_df = pd.read_csv(csv_data)
    print(voters_df.head())
else:
    print('Failed to retrieve the file. Status code:', response.status_code)


In [ ]:
voters_df.head(5)

## First look at the data

Before doing anything else, it's good practice to check the **shape** of the data (how many voters, how many variables) and the **column names and types**.


In [ ]:
print('Number of voters (rows) and variables (columns):', voters_df.shape)
print()
print('Column names:')
print(list(voters_df.columns))
print()
voters_df.dtypes

In [ ]:
# Quick summary statistics for all numeric columns
voters_df.describe()

Depending on how the file was built, the exact column names might differ slightly. The two cells below automatically pick one numeric column (e.g., something like age) and one categorical column (e.g., something like party or gender) to use in the examples that follow, so the notebook runs no matter the exact column names. Once you know your real column names from the output above, feel free to swap them in directly.


In [ ]:
numeric_cols = voters_df.select_dtypes(include='number').columns.tolist()
categorical_cols = voters_df.select_dtypes(include='object').columns.tolist()

demo_num_col = numeric_cols[0]
demo_cat_col = categorical_cols[0] if len(categorical_cols) > 0 else None

print('Numeric columns:', numeric_cols)
print('Categorical (text) columns:', categorical_cols)
print()
print('We will use', repr(demo_num_col), 'as our example numeric column')
print('We will use', repr(demo_cat_col), 'as our example categorical column')

## Columns

To see just one column, access it with the same square bracket notation as for lists `[]`. However, rather than a number, use the column name string, the same as in a dictionary.


In [ ]:
voters_df[demo_num_col]

The column returned from such an indexing is called a `Series` object.


#### Series functions

A few useful functions we can call on a `Series`: `.mean()`, `.median()`, `.min()`, `.max()`, `.value_counts()`.


In [ ]:
print('Mean:', voters_df[demo_num_col].mean())
print('Median:', voters_df[demo_num_col].median())
print('Min:', voters_df[demo_num_col].min())
print('Max:', voters_df[demo_num_col].max())

In [ ]:
if demo_cat_col is not None:
    display(voters_df[demo_cat_col].value_counts())
else:
    print('No categorical column found to demonstrate value_counts().')

## Masks

A **mask** is a `Series` of `True`/`False` values, one per row, telling us whether each row meets a condition.


In [ ]:
threshold = voters_df[demo_num_col].median()
mask = voters_df[demo_num_col] > threshold
mask

To apply the mask, we simply select all rows where the mask holds:


In [ ]:
voters_df[mask]

We can write the resulting DataFrame to a new **view** of the data, so we can work with it directly:


In [ ]:
above_median_df = voters_df[mask]
above_median_df.head()

We can combine several conditions with the `&` (logical AND: both conditions apply) or `|` (logical OR: at least one condition applies) operators. Remember to wrap each condition in parentheses!


In [ ]:
if demo_cat_col is not None:
    first_category = voters_df[demo_cat_col].dropna().unique()[0]
    combined_mask = (voters_df[demo_num_col] > threshold) & (voters_df[demo_cat_col] == first_category)
    voters_df[combined_mask].head()
else:
    print('No categorical column found to demonstrate combined masks.')

## apply()

We can apply a function to an entire `Series`. The most common case is **casting** a column to make sure every entry has the same data type.


In [ ]:
voters_df[demo_num_col].apply(float).head()

However, functions can be arbitrarily complex. You can write your own function and then apply it to a Series.


In [ ]:
def above_or_below_median(value):
    if value > threshold:
        return 'above median'
    else:
        return 'at or below median'

voters_df[demo_num_col].apply(above_or_below_median).head()

### Adding new columns


In [ ]:
voters_df['relative_to_median'] = voters_df[demo_num_col].apply(above_or_below_median)
voters_df.head()

This new column is now listed in the `voters_df` object (but it is *not* added to the file we originally read in — nothing is saved back to GitHub!).


## Correlations

For political science questions, it's often useful to see how numeric variables relate to one another, for example whether age is correlated with turnout.


In [ ]:
voters_df[numeric_cols].corr()

## String Columns

If we have a column containing strings, we can apply a whole range of functions we know from `str`, such as `len()`, `lower()`, `isalpha()`, but also useful things like `contains()`. These are accessed through `.str`.


In [ ]:
if demo_cat_col is not None:
    voters_df[demo_cat_col].str.lower().head()
else:
    print('No string column found to demonstrate .str functions.')

## Rows

To select rows by position, use `.iloc[]`. As with lists, you can also use slices, and even lists of integers.


In [ ]:
voters_df.iloc[0]

In [ ]:
voters_df.iloc[0:5]

# Visualization


### Histograms


In [ ]:
voters_df[demo_num_col].hist()

In order to get a different representation, let's define the size of each `bin`. Adding `;` at the end of the line prevents the output of the text in angular brackets in the cell above.


In [ ]:
voters_df[demo_num_col].hist(bins=20);

In [ ]:
if demo_cat_col is not None:
    voters_df[demo_num_col].hist(by=voters_df[demo_cat_col], sharey=True, figsize=(10, 4));
else:
    print('No categorical column found to split the histogram by.')

### Scatterplotting


In [ ]:
if len(numeric_cols) >= 2:
    voters_df.plot.scatter(x=numeric_cols[0], y=numeric_cols[1], alpha=0.5)
else:
    print('Need at least two numeric columns to make a scatterplot.')

For more specialized visualizations, like a heat map of correlations, we can use `seaborn`:


In [ ]:
plt.figure(figsize=(8, 6))
sb.heatmap(voters_df[numeric_cols].corr(), annot=True, cmap='coolwarm')
plt.show()

For many more plotting options, see [https://pandas.pydata.org/pandas-docs/stable/visualization.html](https://pandas.pydata.org/pandas-docs/stable/visualization.html)


# In-class Exercises:

For each of the following exercises, provide both the Python code and the output it generates. Following the results, please write a few sentences about the interpretation of what the findings mean for political science.


## Exercise 1:

In political science, we often explore how turnout or vote propensity varies across demographic groups. Pick one numeric variable (for example, something related to age or a turnout score) and one categorical variable (for example, gender or party) from the dataset, and calculate the average of the numeric variable for each group of the categorical variable. What does this tell you about who is more likely to be politically engaged in this district?


## Exercise 2:

Political analysts often need to isolate and study specific segments of the electorate. Using a mask, create a new DataFrame that includes only voters meeting a condition you choose (for example, voters above a certain age, or voters registered with a particular party). Then, calculate the average of one or two other variables for this specific subgroup, and compare it to the average for the full sample.
